In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Q1

In [2]:
from datasets import load_dataset
dataset = load_dataset("csv", data_files={"train": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"})
def merge_columns(row):
    return {"combined_text": f"{row['prompt']} {row['A']}"}

updated_dataset = dataset.map(merge_columns)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [3]:
row51_combined_text = updated_dataset["train"][51]["combined_text"]
len(row51_combined_text)

614

# Q2

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print(tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522


# Q3

In [5]:
print(tokenizer.sep_token_id)

102


# Q4

In [6]:
encoded_inputs = tokenizer(
    list(updated_dataset["train"]["prompt"]),
    max_length=128,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)
print(encoded_inputs["input_ids"].shape)

torch.Size([2000, 128])


In [7]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Q6

In [8]:
inputs = tokenizer(dataset["train"]["prompt"][0],return_tensors="pt")
outputs = model(**inputs)
last_hidden_states = outputs.last_hidden_state
print(last_hidden_states.shape)

torch.Size([1, 31, 768])


# Q7

In [9]:
cls_vector = last_hidden_states[0 , 0 , :]
# print(cls_vector[0:5])
sum_5 = cls_vector[0:5].sum()
print(sum_5)

tensor(-1.2001, grad_fn=<SumBackward0>)


# Q8

In [10]:
tokenizer2 = AutoTokenizer.from_pretrained("bert-base-uncased")
model2 = AutoModel.from_pretrained("bert-base-uncased",output_attentions=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
import torch
string1 = "Light-ion fusion is a technique."
encoding = tokenizer2(string1, return_tensors="pt")
tokens = tokenizer2.convert_ids_to_tokens(encoding["input_ids"][0])
fusion_index = tokens.index("fusion")
print(fusion_index)

with torch.no_grad():
    outputs = model2(**encoding)

last_layer_attentions = outputs.attentions[-1]
cls_to_fusion_weight = last_layer_attentions[0, 0, 0, fusion_index].item()

print(f"Token list: {tokens}")
print(f"Index of 'fusion': {fusion_index}")
print(f"Exact attention weight: {cls_to_fusion_weight:.4f}")

4
Token list: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Index of 'fusion': 4
Exact attention weight: 0.1025


# Q9

In [12]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

prompt_text = dataset["train"]["prompt"][0]
option_b_text = dataset["train"]["B"][0]

embedding_prompt = model.encode(prompt_text, convert_to_tensor=True)
embedding_option_b = model.encode(option_b_text, convert_to_tensor=True)

similarity_tensor = util.cos_sim(embedding_prompt, embedding_option_b)

similarity_score = similarity_tensor.item()
print(f"Similarity score: {similarity_score:.4f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Similarity score: 0.7658


# Q10

### MiniLM Pipeline

In [23]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
options_keys = ['A', 'B', 'C', 'D', 'E']

def get_preds(row):
    prompt = row['prompt']
    options_text = [row[key] for key in options_keys]
    prompt_embedding = model.encode(prompt, convert_to_tensor=True)
    options_embeddings = model.encode(options_text, convert_to_tensor=True)
    cos_sim = util.cos_sim(prompt_embedding,options_embeddings)[0]
    top_indices = torch.topk(cos_sim,k=5,dim=-1).indices.tolist()
    top_preds = [options_keys[idx] for idx in top_indices]
    return top_preds

qidslm = set()
map3 = 0
for i in range(len(dataset['train'])):
    correct_label = dataset['train'][i]['answer']
    preds = get_preds(dataset['train'][i])
    idx = preds.index(correct_label)
    if idx < 3:
        ap3 = 1/(1+idx)
        qidslm.add(i)        
    else:
        ap3 = 0
    map3 += ap3
map3 = map3/len(dataset['train'])
print(map3)   

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.42308333333333487


In [25]:
len(qidslm)

1294

### TfidF Pipeline

In [31]:
qidstf = set()
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
label_map = {'A':0,'B':1,'C':2,'D':3,'E':4}

for row in dataset['train']:
    question_id = row['id']
    question = row['prompt']
    correct_answer = label_map[row['answer']]
    options = [row['A'], row['B'], row['C'], row['D'], row['E']]
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform([question] + options)
    q_vector = tfidf_matrix[0]
    opt_vectors = tfidf_matrix[1:]
    scrs = cosine_similarity(q_vector, opt_vectors).flatten()
    topidx = np.argsort(scrs)[::-1].tolist()
    idx = topidx.index(correct_answer)
    if idx >= 3:
        qidstf.add(question_id-1)
len(qidstf)    

925

In [33]:
len(qidstf & qidslm)

488

# Q11

In [14]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification")
prompt_text = dataset["train"]["prompt"][1]
candidate_labels = [
    str(dataset["train"]["A"][1]),
    str(dataset["train"]["B"][1]),
    str(dataset["train"]["C"][1])
]

result = classifier(prompt_text, candidate_labels=candidate_labels, multi_label=False)

top_option = result["labels"][0]
top_score = result["scores"][0]

# print(f"Option: {top_option}")
print(f"Score: {top_score:.4f}")

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Score: 0.4575


# Q12

In [15]:
probs_wo_multilable = result["scores"]
sumwo = sum(probs_wo_multilable)

result_multi_lable = classifier(prompt_text, candidate_labels=candidate_labels, multi_label=True)
probs_w_multilable = result_multi_lable["scores"]
sumw = sum(probs_w_multilable)

diff = sumwo-sumw

print(round(diff,4))

0.9995


# Q13

In [16]:
q_string = f"Question: {dataset["train"]["prompt"][0]}. Is the correct answer A: {dataset["train"]["A"][0]} or B: {dataset["train"]["B"][0]}? Answer with just the letter A or B."

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

inputs = tokenizer(q_string, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=5)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

B
